# 01 — Exploratory Data Analysis (EDA)

This notebook inspects the PlantVillage folders, counts images per class, checks readable files, and saves a summary. It does not modify the source dataset.

In [3]:

from pathlib import Path
import random
import numpy as np
from PIL import Image
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

ROOT = Path("/Users/thakkifilyas/Desktop/DL_Project/plant-disease-repo")
DATASET_DIR = Path("/Users/thakkifilyas/Desktop/DL_Project/PlantVillage")

IMAGE_EXTENSIONS = {
    ".jpg", ".jpeg", ".png",
    ".bmp", ".gif", ".tif", ".tiff"
}

print("Project root:", ROOT)
print("Dataset:", DATASET_DIR)

assert DATASET_DIR.is_dir(), f"Dataset folder not found: {DATASET_DIR}"

rows = []
corrupt = []

for class_dir in sorted(p for p in DATASET_DIR.iterdir() if p.is_dir()):
    for path in sorted(class_dir.rglob("*")):

        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS:

            row = {
                "path": str(path.resolve()),
                "class": class_dir.name
            }

            try:
                with Image.open(path) as im:
                    row.update(
                        width=im.width,
                        height=im.height,
                        mode=im.mode
                    )

                rows.append(row)

            except Exception as exc:
                corrupt.append({
                    "path": str(path),
                    "error": str(exc)
                })

df = pd.DataFrame(rows)

print("Classes:", df["class"].nunique())
print("Images:", len(df))
print("Corrupt:", len(corrupt))

display(
    df.groupby("class")
      .size()
      .sort_values(ascending=False)
      .to_frame("images")
)


out = ROOT / "results" / "eda"
out.mkdir(parents=True, exist_ok=True)

df.to_csv(out / "image_inventory.csv", index=False)

pd.DataFrame(corrupt, columns=["path", "error"]).to_csv(
    out / "corrupt_images.csv",
    index=False
)

print("EDA completed successfully!")
print("Results saved to:", out)


Project root: /Users/thakkifilyas/Desktop/DL_Project/plant-disease-repo
Dataset: /Users/thakkifilyas/Desktop/DL_Project/PlantVillage


KeyboardInterrupt: 

**Viva note:** EDA only validates the files. Resizing, normalization, and augmentation happen in the later notebooks.